# Lesson 3.8g — 给臂排名加误差棒（seed 扫描）

3.8.6 的六个臂每个只训了**一次**（`SEED = 0`）。那个 `val_chunk` 是**一个随机变量的一次抽样**，
随机性来自三处：

1. **权重初始化**；
2. **batch 顺序**；
3. **浮点归约顺序**（这正是换到 GPU 后同 seed 的数字也在第二位变化的原因）。

所以 `E3c − E3b = −0.0013`（GPU）/ `−0.0029`（CPU）这个差**无法解释**：它可能是语言条件真的更好，
也可能只是这次抽样的运气。

**扫描**就是把同一个臂换几个 init seed 各训一次，报 **mean ± range**，用来看臂间差是否**超过**
seed 间的波动。

### 扫描范围：4 个臂 × 3 个 seed = 12 次训练

| 臂 | 输入 | 判据 |
|---|---|---|
| **E1b** | `p+g`, hidden 1139 | 与 E2 配对 → **Δ_vision 的误差棒** |
| **E2** | `I+p+g` | 同上 |
| **E3b** | `I+p+task-ID` (L0) | 与 E3c 配对 → **3.8.4.4 的"L0 封顶 L1"** |
| **E3c** | `I+p+ℓ` (L1) | 同上 |

省掉 `E1a`（只是未匹配对照，已知是 2.045× 容量差）与 `E3a`（接线对照，预期 Δ ≈ 0，
已在两个设备上各自验证过）。

### 一条不可违反的纪律：`SPLIT_SEED` 固定为 42

**只有 init seed 变，split seed 不变。** 否则你会同时改变"留出哪些 episode"——验证集本身变了，
比较就没有意义。3.8.6 里正是把两个 seed 混用，导致留出的 episode 从 4 变成 2，
直到一条断言才抓出来。

### 预注册的判据（先写死，再看数）

$$\Delta_{E3} = \overline{\text{E3c}} - \overline{\text{E3b}}$$

- 若 $|\Delta_{E3}|$ **大于**两者 range 的并集宽度 → 差异超过 seed 噪声，可以报告方向；
- 若 $\Delta_{E3}$ **落在**两者的 range 之内 → 只能写 **"不可区分，与信息等价一致"**（3.8.4.4）。

**诚实的限度**：验证集只有 128 个样本，而且 `best_val` 是在**验证集上挑出来的**（乐观偏置）。
所以 3-seed 扫描**只能给出 seed 噪声的量级**，不能把一个小效应变成强证据。要更强需要更多验证
样本，或用一个独立的 selection split 来挑 checkpoint。

## 运行说明

- 共 **12 次训练**：4 个臂 × 3 个 init seed，**一个 (arm, seed) 一个 cell**，所以中断只损失
  当前那一次。
- 用 `scripts/run_notebook_observable.py` 逐 cell 执行可看到每次训练的心跳；直接
  `Run All` 也可以。
- **同一次扫描必须在同一台设备上跑完**——跨设备数字会在第二位分叉（见 `notes/concepts.md`
  的 "Ablation devices"）。preamble 会打印 `device`。
- 顺序依赖只在本书内。`Kernel → Restart Kernel and Run All Cells` 即可。
- 跑完由我来写读数 markdown（按上面的预注册判据）。

In [1]:
# 前置：契约 scripts/mml_contract.py，模型/训练/指标 scripts/mml_policy.py —— 都是单一实现。
import logging
import sys
import warnings
from pathlib import Path

import numpy as np
import torch

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / ".git").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "scripts"))
import mml_contract as mmc
import mml_policy as mp

logging.getLogger("mani_skill").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", message=".*NVML.*")
warnings.filterwarnings("ignore", message=".*CUDA initialization.*")
warnings.filterwarnings("ignore", message=".*cudaGetDeviceCount.*")

VOCAB, T_TXT, H, ACTION_DIM, GRIPPER = mmc.VOCAB, mmc.T_TXT, mmc.H, mmc.ACTION_DIM, 7
EPOCHS = 150

# 两个 seed，绝不能混：
#   SPLIT_SEED 固定 42 —— 决定留出哪些 episode，整个扫描期间**不变**
#   INIT_SEED  扫描的变量 —— 只影响权重初始化与 batch 顺序
SPLIT_SEED = 42
SEEDS = (0, 1, 2)

datasets = mmc.load_datasets()
raw = mp.build_dataset(datasets, H=H)
train_mask, val_mask, held = mp.episode_split(datasets, raw, seed=SPLIT_SEED)
assert held == {"PickCube-v1": [4], "PushCube-v1": [4]}, held
norm = mp.fit_normalization(datasets, held)
data = mp.apply_normalization(raw, norm)

print(f"torch {torch.__version__} | device {mp.DEVICE}")
print(f"samples {len(raw['start_of'])} | train {train_mask.sum()} | val {val_mask.sum()} | held {held}")
print(f"SPLIT_SEED {SPLIT_SEED} (fixed) | INIT_SEED {SEEDS} | epochs {EPOCHS} | H {H}")

mean_chunk = data["action_chunk"][train_mask].reshape(-1, H * ACTION_DIM).mean(0)
_mean_pred = np.broadcast_to(mean_chunk.reshape(H, ACTION_DIM), (int(val_mask.sum()), H, ACTION_DIM))
BASELINE = float(((data["action_chunk"][val_mask] - _mean_pred) ** 2).mean())
print(f"mean-action baseline (val, normalised) = {BASELINE:.6f}")


def run_arm(label, seed, **kw):
    """Train one arm at one init seed. The split is NOT a parameter -- it is fixed above.

    Only the init seed varies, so every run sees exactly the same held-out episodes. Mixing
    the two seeds is what silently moved the held-out episode from 4 to 2 in 3.8.6.
    """
    torch.manual_seed(seed)
    model = mp.MultimodalPolicy(vocab_size=len(VOCAB), t_txt=T_TXT, **kw)
    torch.manual_seed(seed)
    tl = torch.utils.data.DataLoader(mp.DictDataset(data, train_mask), batch_size=32, shuffle=True)
    vl = torch.utils.data.DataLoader(mp.DictDataset(data, val_mask), batch_size=64, shuffle=False)
    res = mp.train_model(model, tl, vl, epochs=EPOCHS, lr=1e-3, seed=seed, verbose_every=0)
    p_h0, t_h0 = mp.predict(model, data, val_mask, horizon=0)
    out = dict(label=label, seed=seed, params=mp.count_params(model),
               val_chunk=res["best_val"], val_h0=mp.mse(p_h0, t_h0),
               gripper=mp.gripper_sign_accuracy(p_h0, t_h0), best_epoch=res["best_epoch"])
    print(f"  {label:<4} seed {seed}  val_chunk {out['val_chunk']:.6f}  val_h0 {out['val_h0']:.6f}"
          f"  grip {out['gripper']:.1%}  params {out['params']:,}  (best epoch {out['best_epoch']})",
          flush=True)
    return out


RESULTS = {}

# 四个臂的 kwargs 与 3.8.6 完全一致，只把 init seed 变成变量
ARMS = {
    "E1b": dict(use_image=False, use_language=False, fusion_hidden=1139),
    "E2":  dict(use_language=False),
    "E3b": dict(use_goal=False, language_mode="task_id"),
    "E3c": dict(use_goal=False, language_mode="tokens"),
}

torch 2.11.0+cu128 | device cuda
samples 637 | train 509 | val 128 | held {'PickCube-v1': [4], 'PushCube-v1': [4]}
SPLIT_SEED 42 (fixed) | INIT_SEED (0, 1, 2) | epochs 150 | H 8
mean-action baseline (val, normalised) = 0.597227


## E1b — `p+g`，容量匹配（hidden 1139）

与 E2 配对使用：`Δ_vision = E1b − E2`。E1b 是**容量对照**，所以它的误差棒决定了
"视觉增益"要多大才算超过容量与 seed 的双重噪声。

In [2]:
RESULTS[("E1b", 0)] = run_arm("E1b", seed=0, **ARMS["E1b"])

  E1b  seed 0  val_chunk 0.102373  val_h0 0.072886  grip 99.2%  params 511,635  (best epoch 39)


In [3]:
RESULTS[("E1b", 1)] = run_arm("E1b", seed=1, **ARMS["E1b"])

  E1b  seed 1  val_chunk 0.068587  val_h0 0.050013  grip 99.2%  params 511,635  (best epoch 35)


In [4]:
RESULTS[("E1b", 2)] = run_arm("E1b", seed=2, **ARMS["E1b"])

  E1b  seed 2  val_chunk 0.085551  val_h0 0.071575  grip 98.4%  params 511,635  (best epoch 10)


## E2 — `I+p+g`

与 E1b 配对的另一半。3.8.6 里它是匹配后 `Δ_vision = +0.014489`（CPU）/ `+0.014186`（GPU）。

In [5]:
RESULTS[("E2", 0)] = run_arm("E2", seed=0, **ARMS["E2"])

  E2   seed 0  val_chunk 0.088348  val_h0 0.068689  grip 93.8%  params 511,712  (best epoch 25)


In [6]:
RESULTS[("E2", 1)] = run_arm("E2", seed=1, **ARMS["E2"])

  E2   seed 1  val_chunk 0.074864  val_h0 0.055804  grip 99.2%  params 511,712  (best epoch 28)


In [7]:
RESULTS[("E2", 2)] = run_arm("E2", seed=2, **ARMS["E2"])

  E2   seed 2  val_chunk 0.084375  val_h0 0.066352  grip 99.2%  params 511,712  (best epoch 23)


## E3b — `I+p+task-ID`（L0）

每个任务一个 embedding。3.8.4.4 预测它是 E3c 的**上界**，所以 E3c 超过它意味着实验坏了
（C 从别处读到了任务身份）。

In [8]:
RESULTS[("E3b", 0)] = run_arm("E3b", seed=0, **ARMS["E3b"])

  E3b  seed 0  val_chunk 0.036425  val_h0 0.026769  grip 99.2%  params 511,648  (best epoch 40)


In [9]:
RESULTS[("E3b", 1)] = run_arm("E3b", seed=1, **ARMS["E3b"])

  E3b  seed 1  val_chunk 0.045598  val_h0 0.039166  grip 100.0%  params 511,648  (best epoch 76)


In [10]:
RESULTS[("E3b", 2)] = run_arm("E3b", seed=2, **ARMS["E3b"])

  E3b  seed 2  val_chunk 0.043625  val_h0 0.041542  grip 99.2%  params 511,648  (best epoch 32)


## E3c — `I+p+ℓ`（L1）

每个词一个 embedding + masked mean pool。这是语言臂，也是 `T2/T3/drop` 三个反事实测试
所作用的那个臂（T2 必须落在 float32 舍入里）。

In [11]:
RESULTS[("E3c", 0)] = run_arm("E3c", seed=0, **ARMS["E3c"])

  E3c  seed 0  val_chunk 0.034067  val_h0 0.021755  grip 99.2%  params 512,288  (best epoch 90)


In [12]:
RESULTS[("E3c", 1)] = run_arm("E3c", seed=1, **ARMS["E3c"])

  E3c  seed 1  val_chunk 0.043474  val_h0 0.028074  grip 99.2%  params 512,288  (best epoch 76)


In [13]:
RESULTS[("E3c", 2)] = run_arm("E3c", seed=2, **ARMS["E3c"])

  E3c  seed 2  val_chunk 0.031756  val_h0 0.022467  grip 99.2%  params 512,288  (best epoch 101)


## 汇总

只打印数据。判据见开头的预注册段落。

In [14]:
missing = [(a, s) for a in ARMS for s in SEEDS if (a, s) not in RESULTS]
assert not missing, f"these runs have not happened yet: {missing}"

arms = list(ARMS)
print(f"{'arm':<6} {'n':>2} {'mean':>10} {'min':>10} {'max':>10} {'range':>10} {'grip mean':>10}")
print("-" * 66)
summary = {}
for a in arms:
    v = np.array([RESULTS[(a, s)]["val_chunk"] for s in SEEDS])
    g = np.array([RESULTS[(a, s)]["gripper"] for s in SEEDS])
    summary[a] = v
    print(f"{a:<6} {len(v):>2} {v.mean():>10.6f} {v.min():>10.6f} {v.max():>10.6f} "
          f"{v.max() - v.min():>10.6f} {g.mean():>9.1%}")

print()
print(f"  {'comparison':<26} {'mean':>11} {'per-seed':>34}")
print("  " + "-" * 74)
for a, b, label in (("E1b", "E2", "Delta_vision = E1b - E2"),
                    ("E3b", "E3c", "Delta_E3     = E3c - E3b")):
    d = np.array([RESULTS[(a, s)]["val_chunk"] - RESULTS[(b, s)]["val_chunk"] for s in SEEDS])
    print(f"  {label:<26} {d.mean():>+11.6f} {str(np.round(d, 6).tolist()):>34}")

print()
print(f"  {'pair':<10} {'overlap?':>10}   seed range of each side")
for a, b, _ in (("E1b", "E2", ""), ("E3b", "E3c", "")):
    lo = max(summary[a].min(), summary[b].min())
    hi = min(summary[a].max(), summary[b].max())
    overlap = "OVERLAP" if lo <= hi else "DISJOINT"
    print(f"  {a}/{b:<7} {overlap:>10}   {a} [{summary[a].min():.6f}, {summary[a].max():.6f}]"
          f"   {b} [{summary[b].min():.6f}, {summary[b].max():.6f}]")

print()
print(f"mean-action baseline = {BASELINE:.6f}")

arm     n       mean        min        max      range  grip mean
------------------------------------------------------------------
E1b     3   0.085504   0.068587   0.102373   0.033787     99.0%
E2      3   0.082529   0.074864   0.088348   0.013484     97.4%
E3b     3   0.041883   0.036425   0.045598   0.009173     99.5%
E3c     3   0.036432   0.031756   0.043474   0.011718     99.2%

  comparison                        mean                           per-seed
  --------------------------------------------------------------------------
  Delta_vision = E1b - E2      +0.002975    [0.014025, -0.006277, 0.001176]
  Delta_E3     = E3c - E3b     +0.005450     [0.002358, 0.002124, 0.011869]

  pair         overlap?   seed range of each side
  E1b/E2         OVERLAP   E1b [0.068587, 0.102373]   E2 [0.074864, 0.088348]
  E3b/E3c        OVERLAP   E3b [0.036425, 0.045598]   E3c [0.031756, 0.043474]

mean-action baseline = 0.597227


## 读数

设备 `cuda`，`SPLIT_SEED = 42` 固定，4 个臂 × 3 个 init seed。

| 臂 | 输入 | n | mean | min | max | **range** | grip |
|---|---|---|---|---|---|---|---|
| E1b | `p+g` hidden 1139 | 3 | 0.085504 | 0.068587 | 0.102373 | **0.033787** | 99.0% |
| E2 | `I+p+g` | 3 | 0.082529 | 0.074864 | 0.088348 | 0.013484 | 97.4% |
| E3b | `I+p+task-ID` | 3 | 0.041883 | 0.036425 | 0.045598 | 0.009173 | 99.5% |
| E3c | `I+p+ℓ` | 3 | 0.036432 | 0.031756 | 0.043474 | 0.011718 | 99.2% |

### 预注册判据的结果：两对**都是 OVERLAP**

| 对比 | mean Δ | 逐 seed | 判定 |
|---|---|---|---|
| `Δ_vision = E1b − E2` | **+0.002975** | `[+0.014025, −0.006277, +0.001176]` | **OVERLAP**，且**符号在 seed 间翻转** |
| `Δ_E3 = E3c − E3b` | **+0.005450** | `[+0.002358, +0.002124, +0.011869]` | **OVERLAP**，但三个 seed **同号** |

**所以 3.8.6 的单 seed 臂排名不成立。** 逐条：

1. **`Δ_vision` 没被建立，而且比"不显著"更糟。** 3.8.6 的单 seed 给出 `+0.014186`（GPU）；
   本扫描里 seed 0 给出 `+0.014025` —— **同一个抽样**。但换 seed 后符号变成 `−0.006277`
   （state-only **赢**），均值只剩 `+0.002975`。那个 `+0.014` 是**一次幸运抽样**，不是图像的信息量。

2. **`Δ_E3` 的符号反而回到了理论预测的方向。** 3.8.6 的单 seed 给 `−0.0029`（CPU）/ `−0.0013`（GPU），
   即 **C 比 B 好**，与 3.8.4.4 的"L0 封顶 L1"相反（当时我说在噪声内）。扫描的**三个 seed 全部为正**
   `[+0.0024, +0.0021, +0.0119]`，即 **C 比 B 差** —— **正是 L0 封顶的方向**。
   量级仍未被建立，但**方向与理论一致了**。

### 扫描真正建立的结论是**另一条**，而且是唯一过判据的

E2 与 E3c **容量相同**（融合维都是 416；参数量 511,712 vs 512,288，差 0.1%），
用**同样的 image + proprio**，只把连续的 `goal [3]` 换成语言 / 任务索引：

$$\text{E2 }(I+p+g):\ 0.0825\ \,[0.0749,\ 0.0883] \qquad\text{vs}\qquad \text{E3c }(I+p+\ell):\ 0.0364\ \,[0.0318,\ 0.0435]$$

**两侧 seed range 不相交，误差差 2.27×**（E3b 同理：`0.0419 [0.0364, 0.0456]`，与 E2 也不相交）。

$$\Rightarrow\quad \textbf{同样的信息，不同的编码，可学性差别很大。}$$

`goal` 与任务索引在这份 pool 上**信息等价**（$H(\ell \mid g) = 0$，3.8.4.6），
但"从 3 维连续实数里阈值出任务"比"读一个离散索引"难学得多。
这把 3.8.4.4 的 L0 / L1 结论**推广了一步**：**表示的选择不只关乎信息量，也关乎可学性。**

### 一个反讽的方法学发现

**E1b 是噪声最大的臂**：range `0.033787`，是 E2（`0.013484`）的 **2.5 倍**，比另外三个臂大 3 倍。
而 E1b 正是**容量匹配对照**本身：

> 容量匹配让我们**能**比较，代价是**对照本身变得最不稳定**——更宽的 MLP 加 509 个样本。
> 所以 `Δ_vision` 恰好是容量匹配让它**最不可靠**的那个对比。

### 诚实限度

- 3 个 seed 给出的是**噪声量级**，不是置信区间；range 本身只是 3 个点的极差。
- 验证集只有 128 个样本，且 `best_val` 是在**验证集上挑出来的**（乐观偏置）。
- E1b 的大 range 让 `Δ_vision` 的误差棒异常宽。若真要认真测视觉增益，应先处理对照的**稳定性**
  （固定对照宽度，或改用独立的 selection split 来挑 checkpoint），而不是继续加 seed。
- 上面所有数字都来自**同一台设备**；跨设备对照（见 3.8.6）只能用作稳健性检查。

## 自检

1. 为什么 `val_chunk` 是一个随机变量？列出至少三个方差来源。
2. 扫描时为什么 `SPLIT_SEED` 必须固定、只有 init seed 变？如果两个都变会发生什么？
3. 如果 `Δ_E3` 的 mean 是 `−0.0013`，而 E3b 的 range 是 `[0.0360, 0.0375]`、E3c 是
   `[0.0345, 0.0362]`，你能得出什么结论？能得出"C 更好"吗？
4. 为什么"3 个 seed"仍然**不足以**把一个小效应变成强证据？还缺什么？
5. `best_val` 是在验证集上挑出来的，这对误差棒有什么影响？更干净的做法是什么？